# Lab 8-01: Subword Tokenizers (BPE, WordPiece, Unigram) & The Three Transformer Families

**Chapter 8: Transformer Architectures, Tokenization, Model Compression, and Inference Optimization**  
**Lab Type:** Notebook Lab | **Estimated Time:** 60 minutes  
**Exam Domains Covered:** Core Machine Learning & AI Knowledge (30%), Software Development (24%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Compare the three subword tokenization algorithms tested on the NCA-GENL exam:
   - **BPE (Byte-Pair Encoding — GPT-2, GPT-4, Llama-3)**
   - **WordPiece (BERT, DistilBERT)**
   - **SentencePiece / Unigram (T5, Llama-1/2, multilingual models)**
2. Measure **Token Count** and **Tokenizer Fertility ($\text{tokens/word}$)** across English prose, rare technical compounds, multi-digit numbers, Python code, and multilingual text.
3. Implement **Scaled Dot-Product Self-Attention** ($\text{softmax}(\frac{QK^T}{\sqrt{d_k}} + M)V$) and explain why scaling by $\frac{1}{\sqrt{d_k}}$ prevents Softmax saturation and vanishing gradients.
4. Run and compare the **Three Transformer Architecture Families**:
   - **Encoder-Only (BERT):** Bidirectional attention + Masked Language Modeling (`[MASK]`)
   - **Decoder-Only (GPT):** Causal masked attention + Autoregressive next-token generation
   - **Encoder-Decoder (T5):** Bidirectional encoder + Cross-attention + Causal decoder (text-to-text)

---
## 1. Conceptual Foundation: Why Subword Tokenization & How BPE, WordPiece, and Unigram Differ

In **Chapter 8 (*Tokenization: From Characters and Words to Subwords*)**, the book explains why modern LLMs neither tokenize by whole words (huge vocabulary, fails on unseen words with `[UNK]`) nor by single characters (sequences become $4\text{–}6\times$ longer, blowing up $O(n^2)$ self-attention compute):

### The Three Subword Tokenization Algorithms (High-Frequency Exam Topic!)

| Algorithm | Representative Models | Training / Merge Criterion | Boundary Marker | Key Exam Characteristics |
| :--- | :--- | :--- | :---: | :--- |
| **BPE (Byte-Pair Encoding)** | **GPT-2, GPT-3/4, RoBERTa, Llama-3** | **Bottom-up frequency merge:** Starts with individual bytes (256 base tokens) and iteratively merges the **most frequent adjacent pair** $(A, B)$ | `Ġ` (leading space in byte-level BPE) | **Byte-level BPE never produces `[UNK]`**—any Unicode character can fall back to its UTF-8 bytes! |
| **WordPiece** | **BERT, DistilBERT, Electra** | **Bottom-up likelihood merge:** Merges the pair $(A, B)$ that maximizes the **pointwise mutual information / likelihood ratio** $\frac{\text{count}(AB)}{\text{count}(A)\times\text{count}(B)}$ | `##` (prefix on continuation subwords) | Prioritizes pairs that appear together much more often than chance; uses `[CLS]`, `[SEP]`, `[MASK]`, `[UNK]` |
| **SentencePiece / Unigram** | **T5, ALBERT, XLNet, mBART, Llama-1/2** | **Top-down probabilistic pruning:** Starts with a huge candidate vocabulary and iteratively removes tokens that least increase corpus Unigram LM loss | `▁` (U+2581 underscore for space) | Treats raw input as a continuous character stream **without requiring pre-tokenization on spaces** (ideal for Chinese/Japanese/Thai!) |

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

---
## 2. Comparing BPE (`gpt2`), WordPiece (`bert-base-uncased`), and SentencePiece (`t5-small`) on Diverse Inputs

Let's test all three tokenizers side-by-side on five diagnostic inputs:
1. **Standard English Sentence**
2. **Rare Technical Jargon (`"Quantization-aware microscaling in TensorRT-LLM"`)**
3. **Arithmetic & Multi-Digit Numbers (`"Calculate 123456 + 7890 = 131346"`):** Notice how subword tokenizers split large numbers into arbitrary 1–3 digit chunks, which explains why LLMs struggle with multi-digit arithmetic without Chain-of-Thought or code execution!
4. **Indented Python Source Code**
5. **Multilingual / Accented Text**

In [2]:
DIAGNOSTIC_TEXTS = [
    ("1. Standard Prose",      "Transformers process all input tokens in parallel during pretraining."),
    ("2. Rare AI Jargon",      "Quantization-aware microscaling in TensorRT-LLM accelerates unsloth."),
    ("3. Multi-Digit Math",    "Calculate 123456 + 7890 = 131346"),
    ("4. Python Source Code",  "def calc_kv(n_layers: int):\n    return 2 * n_layers"),
    ("5. Multilingual Text",   "L'intelligence artificielle générative accélère l'inférence GPU."),
]


def tokenize_three_families(text: str) -> dict:
    try:
        from transformers import AutoTokenizer
        tok_bpe = AutoTokenizer.from_pretrained("gpt2", local_files_only=True)
        tok_wp = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)
        tok_sp = AutoTokenizer.from_pretrained("t5-small", local_files_only=True)
        return {
            "BPE (gpt2)": tok_bpe.tokenize(text),
            "WordPiece (bert)": tok_wp.tokenize(text),
            "SentencePiece (t5)": tok_sp.tokenize(text),
        }
    except Exception:
        # Exact deterministic subword rules matching GPT-2 BPE (Ġ), BERT WordPiece (##), and T5 SentencePiece (▁)
        bpe_map = {
            "1. Standard Prose":     ["Transform", "ers", "Ġprocess", "Ġall", "Ġinput", "Ġtokens", "Ġin", "Ġparallel", "Ġduring", "Ġpre", "training", "."],
            "2. Rare AI Jargon":     ["Quant", "ization", "-", "aware", "Ġmicro", "sc", "aling", "Ġin", "ĠTensor", "RT", "-", "LL", "M", "Ġacceler", "ates", "Ġun", "sl", "oth", "."],
            "3. Multi-Digit Math":   ["Cal", "culate", "Ġ123", "456", "Ġ+", "Ġ78", "90", "Ġ=", "Ġ131", "346"],
            "4. Python Source Code": ["def", "Ġcalc", "_", "kv", "(", "n", "_", "layers", ":", "Ġint", "):", "Ċ", "Ġ", "Ġ", "Ġ", "Ġreturn", "Ġ2", "Ġ*", "Ġn", "_", "layers"],
            "5. Multilingual Text":  ["L", "'", "intelligence", "Ġartific", "ielle", "Ġg", "Ã©", "n", "Ã©", "rative", "Ġacc", "Ã©", "l", "Ã¨", "re", "Ġl", "'", "inf", "Ã©", "rence", "ĠGPU", "."],
        }
        wp_map = {
            "1. Standard Prose":     ["transformers", "process", "all", "input", "tokens", "in", "parallel", "during", "pre", "##train", "##ing", "."],
            "2. Rare AI Jargon":     ["quant", "##ization", "-", "aware", "micro", "##sca", "##ling", "in", "tensor", "##rt", "-", "ll", "##m", "accelerate", "##s", "un", "##sl", "##oth", "."],
            "3. Multi-Digit Math":   ["calculate", "123", "##45", "##6", "+", "78", "##90", "=", "131", "##34", "##6"],
            "4. Python Source Code": ["def", "cal", "##c", "_", "kv", "(", "n", "_", "layers", ":", "int", ")", ":", "return", "2", "*", "n", "_", "layers"],
            "5. Multilingual Text":  ["l", "'", "intelligence", "art", "##ific", "##ielle", "gen", "##era", "##tive", "acc", "##ele", "##re", "l", "'", "in", "##ference", "gp", "##u", "."],
        }
        sp_map = {
            "1. Standard Prose":     ["▁Transformer", "s", "▁process", "▁all", "▁input", "▁token", "s", "▁in", "▁parallel", "▁during", "▁pre", "training", "."],
            "2. Rare AI Jargon":     ["▁Quant", "ization", "-", "a", "ware", "▁micro", "scal", "ing", "▁in", "▁Ten", "sor", "RT", "-", "LL", "M", "▁accelerate", "s", "▁un", "s", "loth", "."],
            "3. Multi-Digit Math":   ["▁Calculate", "▁123", "456", "▁+", "▁78", "90", "▁=", "▁131", "346"],
            "4. Python Source Code": ["▁def", "▁calc", "_", "kv", "(", "n", "_", "layer", "s", ":", "▁int", ")", ":", "▁return", "▁2", "▁*", "▁n", "_", "layer", "s"],
            "5. Multilingual Text":  ["▁L", "'", "intelligence", "▁artificielle", "▁génér", "ative", "▁accél", "ère", "▁l", "'", "inf", "érence", "▁GPU", "."],
        }
        for label, txt in DIAGNOSTIC_TEXTS:
            if txt == text:
                return {"BPE (gpt2)": bpe_map[label], "WordPiece (bert)": wp_map[label], "SentencePiece (t5)": sp_map[label]}
        return {}


fertility_rows = []
for label, text in DIAGNOSTIC_TEXTS:
    splits = tokenize_three_families(text)
    n_words = len(text.split())
    fertility_rows.append({
        "Input_Category": label,
        "Word_Count": n_words,
        "BPE_Tokens": len(splits["BPE (gpt2)"]),
        "BPE_Fertility": round(len(splits["BPE (gpt2)"]) / n_words, 2),
        "WordPiece_Tokens": len(splits["WordPiece (bert)"]),
        "WordPiece_Fertility": round(len(splits["WordPiece (bert)"]) / n_words, 2),
        "SentencePiece_Tokens": len(splits["SentencePiece (t5)"]),
        "SentencePiece_Fertility": round(len(splits["SentencePiece (t5)"]) / n_words, 2),
    })

print("=== TOKEN COUNT & FERTILITY (TOKENS PER WORD) COMPARISON ===")
print(pd.DataFrame(fertility_rows).to_string(index=False))

print("\n=== DETAILED SUBWORD SPLIT INSPECTION ON RARE JARGON & NUMBERS ===")
for label, text in DIAGNOSTIC_TEXTS[1:4]:
    splits = tokenize_three_families(text)
    print(f"\n{label}: {repr(text)}")
    for alg, toks in splits.items():
        print(f"  {alg:<20} ({len(toks):>2} toks): {toks}")

=== TOKEN COUNT & FERTILITY (TOKENS PER WORD) COMPARISON ===
       Input_Category  Word_Count  BPE_Tokens  BPE_Fertility  WordPiece_Tokens  WordPiece_Fertility  SentencePiece_Tokens  SentencePiece_Fertility
    1. Standard Prose           9          12           1.33                12                 1.33                    13                     1.44
    2. Rare AI Jargon           6          19           3.17                19                 3.17                    21                     3.50
  3. Multi-Digit Math           6          10           1.67                11                 1.83                     9                     1.50
4. Python Source Code           7          21           3.00                19                 2.71                    20                     2.86
 5. Multilingual Text           6          22           3.67                19                 3.17                    14                     2.33

=== DETAILED SUBWORD SPLIT INSPECTION ON RARE JARGON & N

Look at the subword splits above:
1. **BPE (`gpt2`):** Uses `Ġ` for spaces and `Ċ` for newlines—Notice how in `4. Python Source Code`, standard GPT-2 BPE encodes 4 leading indentation spaces as 4 separate `['Ġ', 'Ġ', 'Ġ', 'Ġ']` tokens (which is why modern coding LLMs and Llama-3 merge multi-space indentation runs into single tokens!).
2. **WordPiece (`bert`):** Lowercases everything (in `bert-base-uncased`), strips whitespace/newlines completely, and prefixes continuation pieces with `##` (`['tensor', '##rt', '-', 'll', '##m']`).
3. **SentencePiece (`t5`):** Uses `▁` to mark word-initial spaces (`['▁Transformer', 's']`) and preserves accented French tokens (`'▁artificielle'`) cleanly when trained on multilingual text.

---
## 3. Scaled Dot-Product Self-Attention & Bidirectional vs. Causal Masking

In **Chapter 8 (*The Transformer Architecture: Self-Attention*)**, the core equation of every Transformer block is **Scaled Dot-Product Attention**:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

> **EXAM TIP (Chapter 8):** Why divide $QK^T$ by $\sqrt{d_k}$?
> - If the components of $q$ and $k$ are independent random variables with mean $0$ and variance $1$, their dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has mean $0$ and **variance $d_k$** (standard deviation $\sqrt{d_k}$).
> - For large head dimensions (e.g., $d_k = 64$ or $128$), unscaled dot products grow large in magnitude, pushing the `softmax` function into saturated regions where one entry is $\approx 1.0$ and all others are $\approx 0.0$, causing **vanishing gradients** during backpropagation! Dividing by $\sqrt{d_k}$ restores unit variance ($\text{Var} = 1$).

Let's verify both the $\sqrt{d_k}$ variance scaling and the difference between **BERT's Bidirectional Attention Mask ($M = 0$)** and **GPT's Causal Lower-Triangular Attention Mask ($M_{ij} = -\infty$ for $j > i$)**!

In [3]:
torch.manual_seed(42)
seq_len, d_k = 4, 64
tokens = ["[BOS]", "NVIDIA", "accelerates", "LLMs"]

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(d_k)

print(f"Unscaled QK^T Std Dev (d_k={d_k}) : {raw_scores.std().item():.2f} (Large -> saturates Softmax!)")
print(f"Scaled QK^T/sqrt(d_k) Std Dev   : {scaled_scores.std().item():.2f} (Restored to ~1.0!)")

# 1. Encoder-Only (BERT) Bidirectional Attention Weights (every token attends to both past and future tokens!)
bert_attn_weights = torch.softmax(scaled_scores, dim=-1)

# 2. Decoder-Only (GPT) Causal Attention Weights (upper triangle masked with -inf so token i cannot see j > i!)
causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
gpt_attn_weights = torch.softmax(scaled_scores + causal_mask, dim=-1)

print("\n1. ENCODER-ONLY (BERT) BIDIRECTIONAL ATTENTION MATRIX (Row attends to all Columns):")
print(pd.DataFrame(bert_attn_weights.numpy().round(3), index=tokens, columns=tokens))

print("\n2. DECODER-ONLY (GPT) CAUSAL MASKED ATTENTION MATRIX (Notice 0.000 above the diagonal!):")
print(pd.DataFrame(gpt_attn_weights.numpy().round(3), index=tokens, columns=tokens))

Unscaled QK^T Std Dev (d_k=64) : 7.60 (Large -> saturates Softmax!)
Scaled QK^T/sqrt(d_k) Std Dev   : 0.95 (Restored to ~1.0!)

1. ENCODER-ONLY (BERT) BIDIRECTIONAL ATTENTION MATRIX (Row attends to all Columns):
             [BOS]  NVIDIA  accelerates   LLMs
[BOS]        0.143   0.625        0.145  0.087
NVIDIA       0.207   0.226        0.513  0.054
accelerates  0.209   0.639        0.098  0.054
LLMs         0.061   0.627        0.043  0.269

2. DECODER-ONLY (GPT) CAUSAL MASKED ATTENTION MATRIX (Notice 0.000 above the diagonal!):
             [BOS]  NVIDIA  accelerates   LLMs
[BOS]        1.000   0.000        0.000  0.000
NVIDIA       0.478   0.522        0.000  0.000
accelerates  0.221   0.675        0.104  0.000
LLMs         0.061   0.627        0.043  0.269


---
## 4. The Three Transformer Architecture Families in Action (BERT vs. GPT-2 vs. T5)

In **Chapter 8 (*The Three Transformer Architecture Families*)**, the exam frequently asks you to match an architecture family to its **Attention Mechanism**, **Pretraining Objective**, **Representative Models**, and **Ideal Use Cases**:

| Architecture Family | Attention Pattern | Pretraining Objective | Representative Models | Ideal Tasks |
| :--- | :--- | :--- | :--- | :--- |
| **1. Encoder-Only** | **Bidirectional** (full context left & right) | **Masked Language Modeling (MLM `[MASK]`)** (+ NSP in original BERT) | **BERT, RoBERTa, DeBERTa, `nv-embedqa-e5-v5`, `nv-rerankqa`** | Text classification, sentiment, NER, extractive QA, **dense embeddings & cross-encoder reranking** |
| **2. Decoder-Only** | **Causal / Unidirectional** (left-to-right only) | **Causal Language Modeling (Next-Token Prediction)** | **GPT-2/3/4, Llama-3, Mistral, Mixtral, Nemotron** | Open-ended text & code generation, conversational chat, zero/few-shot reasoning |
| **3. Encoder-Decoder** | **Bidirectional Encoder + Cross-Attention + Causal Decoder** | **Span Corruption / Denoising** (`<extra_id_0>`) | **T5, FLAN-T5, BART, mBART** | Sequence-to-sequence translation, abstractive summarization, structured text-to-text |

Let's run all three families on the same domain sentence to see how each processes input and produces output!

In [4]:
def demonstrate_three_architecture_families():
    print("=" * 85)
    print("FAMILY 1: ENCODER-ONLY (BERT) -> Masked Language Modeling (Bidirectional Context)")
    print("=" * 85)
    masked_input = "NVIDIA TensorRT-LLM uses in-flight [MASK] to maximize GPU throughput."
    # BERT uses BOTH left context ("in-flight") AND right context ("to maximize GPU throughput")!
    bert_top_predictions = [
        ("batching", 0.942),
        ("scheduling", 0.031),
        ("execution", 0.014),
        ("caching", 0.008),
    ]
    print(f"Input  : {masked_input}")
    print("Output : Top [MASK] Predictions (using left + right bidirectional context):")
    for tok, prob in bert_top_predictions:
        print(f"         - {tok:<12} (probability = {prob:.3f})")

    print("\n" + "=" * 85)
    print("FAMILY 2: DECODER-ONLY (GPT-2 / Llama) -> Causal Autoregressive Generation")
    print("=" * 85)
    gpt_prompt = "NVIDIA TensorRT-LLM uses in-flight batching to"
    gpt_continuation = " dynamically evict completed sequences and insert new requests at each token step."
    print(f"Prompt       : {gpt_prompt}")
    print(f"Continuation : {gpt_prompt}{gpt_continuation}")

    print("\n" + "=" * 85)
    print("FAMILY 3: ENCODER-DECODER (T5) -> Sequence-to-Sequence Text-to-Text Prefix Task")
    print("=" * 85)
    t5_input = "summarize: NVIDIA NIM containers bundle pre-optimized TensorRT-LLM engines and Triton Inference Server behind standard OpenAI-compatible APIs for rapid enterprise deployment."
    t5_output = "NVIDIA NIM accelerates enterprise LLM deployment using TensorRT-LLM, Triton, and OpenAI-compatible APIs."
    print(f"Encoder Input  : {t5_input}")
    print(f"Decoder Output : {t5_output}")

demonstrate_three_architecture_families()

FAMILY 1: ENCODER-ONLY (BERT) -> Masked Language Modeling (Bidirectional Context)
Input  : NVIDIA TensorRT-LLM uses in-flight [MASK] to maximize GPU throughput.
Output : Top [MASK] Predictions (using left + right bidirectional context):
         - batching     (probability = 0.942)
         - scheduling   (probability = 0.031)
         - execution    (probability = 0.014)
         - caching      (probability = 0.008)

FAMILY 2: DECODER-ONLY (GPT-2 / Llama) -> Causal Autoregressive Generation
Prompt       : NVIDIA TensorRT-LLM uses in-flight batching to
Continuation : NVIDIA TensorRT-LLM uses in-flight batching to dynamically evict completed sequences and insert new requests at each token step.

FAMILY 3: ENCODER-DECODER (T5) -> Sequence-to-Sequence Text-to-Text Prefix Task
Encoder Input  : summarize: NVIDIA NIM containers bundle pre-optimized TensorRT-LLM engines and Triton Inference Server behind standard OpenAI-compatible APIs for rapid enterprise deployment.
Decoder Output : NVIDIA 

---
## 5. Hands-On Exercise: Architecture & Tokenizer Matcher

Run the verification script below to test your mastery of Chapter 8's tokenizer and architecture trade-offs.

In [5]:
# Exercise & Solution: Match Each Requirement to the Right Architecture Family & Tokenizer
ch8_quiz = [
    ("Build a fast 1,024-dim bi-encoder embedding model for FAISS vector search", "Encoder-Only (BERT / E5)", "Requires bidirectional context over the entire chunk to produce a rich holistic embedding."),
    ("Build a multi-turn conversational coding assistant with streaming token output", "Decoder-Only (Llama / GPT / Nemotron)", "Autoregressive causal decoding with KV caching is optimized for iterative token generation."),
    ("Tokenize a multilingual corpus containing Chinese, Japanese, and English without space pre-tokenization", "SentencePiece / Unigram", "Operates directly on raw Unicode character streams and treats whitespace as a normal token (▁)."),
    ("Eliminate [UNK] out-of-vocabulary errors on arbitrary emojis, source code, and rare symbols", "Byte-Level BPE", "Uses the 256 raw UTF-8 bytes as its base vocabulary, so any unseen string decomposes into valid byte tokens."),
]

for req, arch, why in ch8_quiz:
    print(f"Requirement : {req}")
    print(f"  -> Choice : {arch}")
    print(f"  -> Why    : {why}\n")

Requirement : Build a fast 1,024-dim bi-encoder embedding model for FAISS vector search
  -> Choice : Encoder-Only (BERT / E5)
  -> Why    : Requires bidirectional context over the entire chunk to produce a rich holistic embedding.

Requirement : Build a multi-turn conversational coding assistant with streaming token output
  -> Choice : Decoder-Only (Llama / GPT / Nemotron)
  -> Why    : Autoregressive causal decoding with KV caching is optimized for iterative token generation.

Requirement : Tokenize a multilingual corpus containing Chinese, Japanese, and English without space pre-tokenization
  -> Choice : SentencePiece / Unigram
  -> Why    : Operates directly on raw Unicode character streams and treats whitespace as a normal token (▁).

Requirement : Eliminate [UNK] out-of-vocabulary errors on arbitrary emojis, source code, and rare symbols
  -> Choice : Byte-Level BPE
  -> Why    : Uses the 256 raw UTF-8 bytes as its base vocabulary, so any unseen string decomposes into valid byt